## Imports

In [1]:
from src.models.U_net import UNet
from src.train import model_trainer
from src.dataset.dataset import ADE20KDataset, make_train_transform, make_val_transform
from src.utils.params import Params, integrate_global_parameters
from src.preprocessing.verify_model import verify_model_file, verify_model_memorization
from src.preprocessing.hyperparameter_estimation import estimate_hyperparameters


c:\Users\Komputer\Documents\Igor\Studia\Sem5\CV\Project3-CV-segmentation-inpainting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.preprocessing.presize_images import process_split

In [8]:
parameters : Params = Params('src/config/uNet_params.json')
model = UNet(in_channels=3, base_channels=64, num_classes=parameters.get("num_classes", 151))
global_parameters = Params()
global_parameters = integrate_global_parameters(global_parameters)
dataset = ADE20KDataset(global_parameters.get("train_image_folder"),
                        global_parameters.get("train_annotation_folder"),
                        transform=make_train_transform(
                            mean = global_parameters.get("mean", (0.485, 0.456, 0.406)),
                            std = global_parameters.get("std", (0.229, 0.224, 0.225))
                            )
                        )
parameters.set("batch_size", 8)
parameters.set("learning_rate", 0.005)
verify_model_memorization(model = model, dataset=dataset, sample_size=8, params=parameters, epochs=200)

2026/01/25 13:36:19 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/01/25 13:36:19 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Training on device: cpu
Using single GPU or CPU
DataLoader settings - batch_size: 8, num_workers: 4, pin_memory: False, prefetch_factor: 2
Epoch 1/200, Train Loss: 9.4051
  Validation: {'CombinedLoss': 21711.78125, 'DiceLoss': 0.9856176376342773, 'mIoU': 0.002061315346509218, 'pixel_accuracy': 0.03547096252441406}
  -> Best model saved with val_loss: 21711.7812
Most recent model saved to ./models/recent_unet.pt
Epoch 2/200, Train Loss: 8.2754
  Validation: {'CombinedLoss': 12636303.0, 'DiceLoss': 0.9796203970909119, 'mIoU': 0.00311078317463398, 'pixel_accuracy': 0.08899116516113281}
Most recent model saved to ./models/recent_unet.pt
Epoch 3/200, Train Loss: 7.3399
  Validation: {'CombinedLoss': 4398037.5, 'DiceLoss': 0.9796977639198303, 'mIoU': 0.0030662084463983774, 'pixel_accuracy': 0.08891868591308594}
Most recent model saved to ./models/recent_unet.pt
Epoch 4/200, Train Loss: 6.6997
  Validation: {'CombinedLoss': 651342.9375, 'DiceLoss': 0.9796979427337646, 'mIoU': 0.00317566725425

2026/01/25 13:38:27 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/01/25 13:38:27 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Epoch 12/200, Train Loss: 4.1370
  Validation: {'CombinedLoss': 67.16259765625, 'DiceLoss': 0.9183497428894043, 'mIoU': 0.0661369264125824, 'pixel_accuracy': 0.29364776611328125}


KeyboardInterrupt: 

In [5]:
from pathlib import Path
process_split(Path(global_parameters.get("validation_image_folder")),
            Path(global_parameters.get("validation_annotation_folder")),
            Path('data/processed/images/validation_320'),
            Path('data/processed/annotations/validation_320'),
            short_side=320)

In [6]:
len(dataset)

20210

In [3]:
estimate_hyperparameters(model=model, model_params_path='src/config/uNet_params.json',
                         dataset=dataset, n_trials=20, sample_size=16, epochs=5)

[I 2026-01-24 22:49:37,090] A new study created in memory with name: no-name-ad8ed582-508e-47ef-bf1b-0393603e847a


  0%|          | 0/20 [00:00<?, ?it/s]2026/01/24 22:49:37 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/01/24 22:49:37 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


{'batch_size': 16, 'learning_rate': 0.0013900708803839257, 'num_epochs': 5, 'optimizer': 'AdamW', 'optimizer_momentum': 0.9, 'optimizer_momentum2': 0.999, 'scheduler': 'StepLR', 'scheduler_step': 7, 'scheduler_gamma': 0.96, 'early_stopping_patience': 15, 'early_stopping_min_delta': 0.0, 'best_model_path': './models/best_unet.pt', 'recent_model_path': './models/recent_unet.pt', 'train_loss_function': 'CombinedLoss', 'validation_loss_functions': ['CombinedLoss', 'DiceLoss'], 'train_loss_params': {'label_smoothing': 0.04020738589099711}, 'validation_loss_params': {'label_smoothing': 0.04020738589099711}, 'num_classes': 151, 'weight_decay': 4.902158431927971e-06, 'train_image_folder': './data/archive/ADEChallengeData2016/images/training/', 'train_annotation_folder': './data/archive/ADEChallengeData2016/annotations/training/', 'validation_image_folder': './data/archive/ADEChallengeData2016/images/validation/', 'validation_annotation_folder': './data/archive/ADEChallengeData2016/annotations/

2026/01/24 22:52:14 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/01/24 22:52:14 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Most recent model saved to ./models/recent_unet.pt
Training complete.


Best trial: 0. Best value: 237.721:   5%|▌         | 1/20 [02:40<50:46, 160.32s/it]2026/01/24 22:52:17 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/01/24 22:52:17 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


[I 2026-01-24 22:52:17,413] Trial 0 finished with value: 237.72142028808594 and parameters: {'lr': 0.0013900708803839257, 'weight_decay': 4.902158431927971e-06, 'ce_weight': 1.8839653701074834, 'dice_weight': 0.4901041005498801, 'label_smoothing': 0.04020738589099711}. Best is trial 0 with value: 237.72142028808594.
{'batch_size': 16, 'learning_rate': 0.0013719744176630528, 'num_epochs': 5, 'optimizer': 'AdamW', 'optimizer_momentum': 0.9, 'optimizer_momentum2': 0.999, 'scheduler': 'StepLR', 'scheduler_step': 7, 'scheduler_gamma': 0.96, 'early_stopping_patience': 15, 'early_stopping_min_delta': 0.0, 'best_model_path': './models/best_unet.pt', 'recent_model_path': './models/recent_unet.pt', 'train_loss_function': 'CombinedLoss', 'validation_loss_functions': ['CombinedLoss', 'DiceLoss'], 'train_loss_params': {'label_smoothing': 0.03205031168375629}, 'validation_loss_params': {'label_smoothing': 0.03205031168375629}, 'num_classes': 151, 'weight_decay': 0.00022802972726513305, 'train_image_

2026/01/24 22:52:49 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/01/24 22:52:49 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
Best trial: 0. Best value: 237.721:   5%|▌         | 1/20 [03:12<1:00:59, 192.59s/it]


[W 2026-01-24 22:52:49,643] Trial 1 failed with parameters: {'lr': 0.0013719744176630528, 'weight_decay': 0.00022802972726513305, 'ce_weight': 1.7200482770213945, 'dice_weight': 0.46737647691621886, 'label_smoothing': 0.03205031168375629} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Komputer\Documents\Igor\Studia\Sem5\CV\Project3-CV-segmentation-inpainting\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\Komputer\Documents\Igor\Studia\Sem5\CV\Project3-CV-segmentation-inpainting\src\preprocessing\hyperparameter_estimation.py", line 66, in objective
    trainer.train_model(train_subset, val_subset, params_obj, optuna_trial=trial)
  File "c:\Users\Komputer\Documents\Igor\Studia\Sem5\CV\Project3-CV-segmentation-inpainting\src\train.py", line 250, in train_model
    train_loss = self.train_1_epoch(optimizer, loss_fn, devi

KeyboardInterrupt: 